In [1]:
#Install sklego
! pip install sklego

In [2]:
#Create a model with binary protected attributes.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklego.metrics import equal_opportunity_score

#Load the dataset
credit_train_df = pd.read_csv("credit-approval-training-data.csv")

#Train the model
trng_features = credit_train_df[["AGE_RANGE",
                                 "INCOME_CATEGORY",
                                 "RACE",
                                 "CREDIT_RATING"]]
trng_target=credit_train_df[["APPROVED"]]

pd.options.mode.chained_assignment = None

#Convert age_range and race to binary
trng_features["AGE_RANGE"] = np.where(trng_features["AGE_RANGE"] == 3,1,0)
trng_features["RACE"] = np.where(trng_features["RACE"] < 3,1,0)
pd.options.mode.chained_assignment = 'warn'

train_X, test_X, train_Y, test_Y = (train_test_split(
    trng_features,trng_target,test_size = 0.30))

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

credit_classifier = GaussianNB()
credit_model = credit_classifier.fit(train_X,train_Y.values.ravel())

approval_predictions = credit_classifier.predict(test_X)
print("Training Classifier Accuracy : ", 
      accuracy_score(test_Y,approval_predictions))

#Compute Equal opportunity score
print("\nequal_opportunity_score for Age Range:", 
      equal_opportunity_score(sensitive_column="AGE_RANGE")
      (credit_classifier, trng_features, trng_target["APPROVED"]))

print("equal_opportunity_score for Race: ", 
      equal_opportunity_score(sensitive_column="RACE")
      (credit_classifier, trng_features, trng_target["APPROVED"]))


Training Classifier Accuracy :  0.8733333333333333

equal_opportunity_score for Age Range: 0.9363246653455736
equal_opportunity_score for Race:  0.8694581280788177


In [3]:
#Load the production dataset
fair_approval_df = pd.read_csv("credit-approval-fair-data.csv")
fair_features = fair_approval_df[["AGE_RANGE",
                                  "INCOME_CATEGORY",
                                  "RACE",
                                  "CREDIT_RATING"]]
fair_target=fair_approval_df[["APPROVED"]]

pd.options.mode.chained_assignment = None

#Convert to binary
fair_features["AGE_RANGE"] = np.where(fair_features["AGE_RANGE"] == 3,1,0)
fair_features["RACE"] = np.where(fair_features["RACE"] < 3,1,0)
pd.options.mode.chained_assignment = 'warn'

#Compute Equal opportunity score for production data
print('equal_opportunity_score for Age Range:', 
      equal_opportunity_score(sensitive_column="AGE_RANGE")
      (credit_classifier, fair_features, fair_target["APPROVED"]))

print('equal_opportunity_score for Race:', 
      equal_opportunity_score(sensitive_column="RACE")
      (credit_classifier, fair_features, fair_target["APPROVED"]))


equal_opportunity_score for Age Range: 0.836940354008627
equal_opportunity_score for Race: 0.5763081395348837
